In [2]:
import os, time, subprocess, pythoncom, psutil
import polars as pl
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
from IPython.display import HTML, display

# ══════════════════════════════════════════════════════════════════════════════
# PATHS & CONFIG
# ══════════════════════════════════════════════════════════════════════════════
first_glob      = os.path.expanduser("~").replace("\\", "/")
PARQUET_PATH    = (f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files"
                   f"/BI_Task/CODE/Resources/excalibur_raw.parquet")
IC_DETAILS_PATH = (f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files"
                   f"/BI_Task/CODE/Resources/IC_HCM_Details_Log.parquet")

DISPLAY_NOTEBOOK = True
SEND_EMAIL       = True

# FIX: removed trailing comma that was creating a tuple
EMAIL_TO = (
    "puneet.suneja@concentrix.com;"
    "kirpan.patar@concentrix.com;"
    "ML.HOC.Expedia.Hierarchy@concentrix.com;"
    "EG_CAI_RAYAH_expedia_global_rtm@concentrix.com"
)

EMAIL_CC = (
    "Varun.Kathuria@concentrix.com;"
    "urmila.chakka1@concentrix.com;"
    "van.tran@concentrix.com;"
    "duonghoangvu.pham@concentrix.com;"
    "VN_HOC_QUANG_vn_hcm_one_exp_wfm@concentrix.com;"
    "ExpediaVN_Training_Team@concentrix.com;"
    "ExpediaVN_QA_Team@concentrix.com;"
    "atul.pathak@concentrix.com"
)

OVERRIDE_DATE = None  # None = auto D-1 | "2026-05-28"

if OVERRIDE_DATE:
    report_date = datetime.strptime(OVERRIDE_DATE, "%Y-%m-%d")
    print(f"OVERRIDE: {OVERRIDE_DATE}")
else:
    report_date = datetime.now() - timedelta(days=1)
    print(f"AUTO D-1 = {report_date.strftime('%Y-%m-%d')}")

report_date_d = report_date.date()
report_date_s = report_date.strftime("%d-%b-%Y")
report_now    = datetime.now()
EMAIL_SUBJECT = f"Expedia VN — IC & Staffing Attainment Report for LGC & NLC as of {report_date_s}"

SITES   = ["VN", "KOL", "PUN", "CAI"]
LOBS    = ["Lodging chat", "Non-Lodging chat"]
LOB_MAP = {"Lodging chat": "LG Chat", "Non-Lodging chat": "NL Chat"}

SITE_MAP = {
    "Concentrix (Ho Chi Minh City)": "VN",
    "Concentrix (Kolkata)":          "KOL",
    "Concentrix (Pune)":             "PUN",
    "Concentrix (Cairo)":            "CAI",
}

IC_TARGET   = 90.0
SA_TARGET   = 95.0
IC_FAIL_RED = 4

DAILY_FROM       = "2026-07-01"
DAILY_TO         = None
N_DAILY_AUTO     = 6
N_WEEKLY_WEEKS   = 8
N_MONTHLY_MONTHS = 6

DAILY_METRICS   = ["IC Pass", "IC Fail", "IC %", "Staffing Attainment %"]
WEEKLY_METRICS  = ["IC Pass", "IC Fail", "IC %", "Staffing Attainment %"]
MONTHLY_METRICS = ["IC Pass", "IC Fail", "IC %", "Staffing Attainment %"]

# ══════════════════════════════════════════════════════════════════════════════
# LOAD PARQUET — excalibur_raw
# ══════════════════════════════════════════════════════════════════════════════
print("Loading excalibur_raw.parquet...")
df_pl = pl.read_parquet(PARQUET_PATH)
df    = df_pl.to_pandas()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()

if "Site_Abbr" not in df.columns:
    df["Site_Abbr"] = df["Site"].map(SITE_MAP).fillna(df["Site"])

df["IC_Pass"] = (df["Interval Compliance (Pct)"] == 1).astype(int)
df["IC_Fail"] = (df["Interval Compliance (Pct)"] == 0).astype(int)
print(f"excalibur: {len(df):,} rows | {df['Date'].min().date()} -> {df['Date'].max().date()}")

# ══════════════════════════════════════════════════════════════════════════════
# LOAD PARQUET — IC HCM Details Log
# ══════════════════════════════════════════════════════════════════════════════
print("Loading IC_HCM_Details_Log.parquet...")
df_ic = pd.read_parquet(IC_DETAILS_PATH)
df_ic["PST_Date"] = pd.to_datetime(df_ic["PST_Date"], errors="coerce").dt.normalize()
df_ic["VNT_Date"] = pd.to_datetime(df_ic["VNT_Date"], errors="coerce").dt.normalize()
df_ic["LOB_disp"] = df_ic["LOB"].map(LOB_MAP).fillna(df_ic["LOB"])
print(f"IC Details: {len(df_ic):,} rows | {df_ic['PST_Date'].min().date()} -> {df_ic['PST_Date'].max().date()}")

# Resolve daily range
if DAILY_TO is None:
    _excalibur_max = df["Date"].max().date()
    daily_to       = _excalibur_max          # FIX: removed - timedelta(days=1)
    print(f"Excalibur max: {_excalibur_max} -> daily_to: {daily_to}")
else:
    daily_to = datetime.strptime(DAILY_TO, "%Y-%m-%d").date()

if DAILY_FROM is None:
    _all_dates = sorted(df[df["Date"].dt.date <= daily_to]["Date"].dt.date.unique())
    daily_from = _all_dates[-N_DAILY_AUTO] if len(_all_dates) >= N_DAILY_AUTO else _all_dates[0]
else:
    daily_from = datetime.strptime(DAILY_FROM, "%Y-%m-%d").date()

print(f"Daily range: {daily_from} -> {daily_to}")
print(f"Subject    : {EMAIL_SUBJECT}")

# ══════════════════════════════════════════════════════════════════════════════
# IC METRICS — dedup per interval, SA uses sum/sum (not mean of ratios)
# ══════════════════════════════════════════════════════════════════════════════
def compute_ic(df_sub: pd.DataFrame) -> dict:
    ic_dedup = (
        df_sub
        .groupby(["Date", "PST_Interval"], as_index=False)
        .agg(ic_val=("Interval Compliance (Pct)", "first"))
        .dropna(subset=["ic_val"])
    )
    total   = len(ic_dedup)
    ic_pass = int((ic_dedup["ic_val"] == 1).sum())
    ic_fail = int((ic_dedup["ic_val"] == 0).sum())
    ic_pct  = round(ic_pass / total * 100, 2) if total > 0 else None

    grp = (
        df_sub
        .groupby(["Date", "PST_Interval"], as_index=False)
        .agg(
            tot_prod=("Total Productive Hours", "max"),
            tot_psp =("MSP site wise",          "sum"),  # sum across sites
        )
    )
    grp = grp[grp["tot_psp"] > 0]
    total_psp  = grp["tot_psp"].sum()
    total_prod = grp["tot_prod"].sum()
    sa_pct = round(total_prod / total_psp * 100, 2) if total_psp > 0 else None

    return {
        "IC Pass": ic_pass,
        "IC Fail": ic_fail,
        "IC %":    ic_pct,
        "Staffing Attainment %": sa_pct,
    }

# ══════════════════════════════════════════════════════════════════════════════
# PIVOT BUILDER
# ══════════════════════════════════════════════════════════════════════════════
def build_pivot(df_in: pd.DataFrame, time_col: str, metrics: list,
                ordered_periods: list = None):
    periods  = ordered_periods if ordered_periods else sorted(df_in[time_col].dropna().unique())
    lob_keys = list(LOB_MAP.values()) + ["Total"]
    records  = {(lob, m): {} for lob in lob_keys for m in metrics}

    for p in periods:
        df_p = df_in[df_in[time_col] == p]
        for lob_raw, lob_key in LOB_MAP.items():
            df_lob = df_p[df_p["LOB"] == lob_raw]
            if df_lob.empty: continue
            m = compute_ic(df_lob)
            for mk in metrics:
                records[(lob_key, mk)][p] = m.get(mk)
        m_all = compute_ic(df_p[df_p["LOB"].isin(LOBS)])
        for mk in metrics:
            records[("Total", mk)][p] = m_all.get(mk)

    rows = []
    for lob in lob_keys:
        for mk in metrics:
            row = {"LOB": lob, "Metric": mk}
            for p in periods:
                row[p] = records.get((lob, mk), {}).get(p)
            rows.append(row)

    return pd.DataFrame(rows), periods

# ══════════════════════════════════════════════════════════════════════════════
# BUILD DAILY PIVOT
# ══════════════════════════════════════════════════════════════════════════════
print("Building Daily pivot...")
df_daily = df[
    (df["Date"].dt.date >= daily_from) &
    (df["Date"].dt.date <= daily_to)
].copy()
df_daily["_D"] = df_daily["Date"].dt.strftime("%Y-%m-%d")
daily_piv, daily_periods = build_pivot(
    df_daily.rename(columns={"_D": "Date_Label"}),
    "Date_Label", DAILY_METRICS
)
print(f"Daily periods: {daily_periods}")

# ══════════════════════════════════════════════════════════════════════════════
# BUILD FAILED INTERVALS — excalibur_raw (Section 2)
# Forecast Productive = sum of MSP site wise across all sites
# Site columns = (Productive Hours - MSP site wise) / 0.5 = variance in heads
# ══════════════════════════════════════════════════════════════════════════════
print("Building Excalibur failed intervals...")
df_fail_ex = df_daily[df_daily["Interval Compliance (Pct)"] == 0].copy()

grp_base = (
    df_fail_ex
    .groupby(["Date","LOB","PST_Interval_Range","PST_Interval"], as_index=False)
    .agg(
        Total_PSP              =("MSP site wise",          "sum"),
        Total_Productive_Hours =("Total Productive Hours", "max"),
    )
)

site_dfs = []
for site in SITES:
    s = (
        df_fail_ex[df_fail_ex["Site_Abbr"] == site]
        .groupby(["Date","LOB","PST_Interval"], as_index=False)
        .agg(prod=("Productive Hours","sum"), msp_site=("MSP site wise","sum"))
    )
    s[site]               = (s["prod"] - s["msp_site"]) / 0.5
    s[f"{site}_raw_fcst"] = s["msp_site"] / 0.5
    s[f"{site}_raw_prod"] = s["prod"] / 0.5
    site_dfs.append(s[["Date","LOB","PST_Interval",
                        site, f"{site}_raw_fcst", f"{site}_raw_prod"]])

fail_site = grp_base.copy()
for s_df in site_dfs:
    fail_site = fail_site.merge(s_df, on=["Date","LOB","PST_Interval"], how="left")

vnt_map = (
    df_fail_ex.groupby(["Date","LOB","PST_Interval"], as_index=False)["VNT_Datetime"].first()
)
vnt_map["VNT_Interval"] = pd.to_datetime(
    vnt_map["VNT_Datetime"], errors="coerce"
).dt.strftime("%H:%M")

fail_site = fail_site.merge(
    vnt_map[["Date","LOB","PST_Interval","VNT_Interval"]],
    on=["Date","LOB","PST_Interval"], how="left"
)
fail_site = fail_site.rename(columns={
    "Total_PSP":              "Total PSP",
    "Total_Productive_Hours": "Total Productive Hours",
})
fail_site["SA %"] = np.where(
    fail_site["Total PSP"] > 0,
    (fail_site["Total Productive Hours"] / fail_site["Total PSP"] * 100).round(2),
    np.nan
)

for site in SITES:
    fail_site[f"{site}_Fcst%"] = np.where(
        (fail_site["Total PSP"] > 0) & fail_site[f"{site}_raw_fcst"].notna(),
        (fail_site[f"{site}_raw_fcst"] / fail_site["Total PSP"] * 100).round(1),
        np.nan
    )
    fail_site[f"{site}_Act%"] = np.where(
        (fail_site["Total Productive Hours"] > 0) & fail_site[f"{site}_raw_prod"].notna(),
        (fail_site[f"{site}_raw_prod"] / fail_site["Total Productive Hours"] * 100).round(1),
        np.nan
    )

fail_site["LOB_disp"] = fail_site["LOB"].map(LOB_MAP).fillna(fail_site["LOB"])
fail_site["Date_str"] = fail_site["Date"].dt.strftime("%Y-%m-%d")
fail_site = fail_site.sort_values(["Date","LOB","PST_Interval"]).reset_index(drop=True)
print(f"Excalibur failed intervals: {len(fail_site)}")

# ══════════════════════════════════════════════════════════════════════════════
# BUILD FAILED INTERVALS — IC HCM Details (Section 3)
# ══════════════════════════════════════════════════════════════════════════════
print("Building IC Details failed intervals...")
df_ic_fail = df_ic[
    (df_ic["IC Status"] == "Missed") &
    (df_ic["PST_Date"].dt.date >= daily_from) &
    (df_ic["PST_Date"].dt.date <= daily_to)
].copy()
df_ic_fail["Date_str"]     = df_ic_fail["PST_Date"].dt.strftime("%Y-%m-%d")
df_ic_fail["VNT_Date_str"] = df_ic_fail["VNT_Date"].dt.strftime("%Y-%m-%d")
df_ic_fail["SA_pct"]       = (df_ic_fail["Staffing Attainment (Pct)"] * 100).round(1)
df_ic_fail = df_ic_fail.sort_values(["PST_Date","LOB","PST_Intervals"]).reset_index(drop=True)
print(f"IC Details failed intervals: {len(df_ic_fail)}")

# ══════════════════════════════════════════════════════════════════════════════
# BUILD WEEKLY & MONTHLY PIVOTS
# ══════════════════════════════════════════════════════════════════════════════
print("Building Weekly pivot...")
df["_Week_Period"] = df["Date"].dt.to_period("W")
df["Week_Label"]   = df["_Week_Period"].apply(
    lambda p: f"{p.start_time.strftime('%m/%d')}~{p.end_time.strftime('%m/%d')}"
    if pd.notna(p) else None
)
week_map = (
    df[df["Date"].dt.date <= daily_to]
    .groupby("_Week_Period", as_index=False)["Week_Label"].first()
    .sort_values("_Week_Period")
)
week_map = week_map[
    week_map["_Week_Period"].apply(lambda p: p.start_time.year) >= daily_to.year
]
recent_week_labels = week_map["Week_Label"].tolist()[-N_WEEKLY_WEEKS:]
df_weekly = df[df["Week_Label"].isin(recent_week_labels)].copy()
weekly_piv, weekly_periods = build_pivot(
    df_weekly.rename(columns={"Week_Label": "Week"}),
    "Week", WEEKLY_METRICS, ordered_periods=recent_week_labels
)
print(f"Weekly ({len(weekly_periods)}): {weekly_periods}")

print("Building Monthly pivot...")
df["Month_Label"] = df["Date"].dt.strftime("%Y-%m")
all_months    = sorted(df[df["Date"].dt.date <= daily_to]["Month_Label"].dropna().unique())
recent_months = all_months[-N_MONTHLY_MONTHS:]
df_monthly    = df[df["Month_Label"].isin(recent_months)].copy()
monthly_piv, monthly_periods = build_pivot(
    df_monthly.rename(columns={"Month_Label": "Month"}),
    "Month", MONTHLY_METRICS, ordered_periods=recent_months
)
print(f"Monthly ({len(monthly_periods)}): {monthly_periods}")

# ══════════════════════════════════════════════════════════════════════════════
# STYLE CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
MET_BG   = "#d4f4e2"; MET_FG  = "#1a5c2a"
WARN_BG  = "#fff3cd"; WARN_FG = "#7a5200"
MISS_BG  = "#fde8ea"; MISS_FG = "#9b1c2a"
HDR_DARK = "#1a3a5c"; HDR_MID = "#1f5c99"
BANNER_C = "#8b0020"

LG_HDR = "#1a6b3a"; LG_ROW = "#f0faf4"; LG_SEP = "#d4edda"
NL_HDR = "#0e3d7a"; NL_ROW = "#f0f5ff"; NL_SEP = "#cce0ff"
TOT_HDR = "#4a5568"; TOT_ROW = "#edf2f7"; TOT_SEP = "#cbd5e0"; TOT_FG = "#2d3748"

SEC_BADGE_BG = "#e6a817"; SEC_BADGE_FG = "#1a1a1a"
FONT = "font-family:Arial,sans-serif;font-size:11px;"
TH_S = (f"{FONT}padding:5px 8px;color:#fff;font-weight:bold;"
        f"white-space:nowrap;text-align:center;"
        f"border:1px solid rgba(255,255,255,0.2);")
TD_S = f"{FONT}padding:4px 8px;border:1px solid #e2e8f0;white-space:nowrap;"

SITE_COLORS = {
    "VN":  {"bar":"#2e86c1","hdr":"#2471a3"},
    "KOL": {"bar":"#1e8449","hdr":"#1a7a42"},
    "PUN": {"bar":"#b7950b","hdr":"#9a7d0a"},
    "CAI": {"bar":"#6c3483","hdr":"#5b2c6f"},
}

CSS = f"""
body{{margin:0;padding:16px;background:#fff;font-family:Arial,sans-serif}}
.t{{border-collapse:collapse;font-size:11px;white-space:nowrap;width:auto}}
.t thead th{{padding:5px 8px;color:#fff;font-weight:bold;
   text-align:center;border:1px solid rgba(255,255,255,0.2)}}
.t tbody td{{padding:4px 8px;border:1px solid #e2e8f0;text-align:right;background:#fff}}
.t tbody td.lbl{{text-align:left}}
.sep-lg td{{background:{LG_SEP};color:{LG_HDR};font-weight:bold;
   padding:3px 10px;font-size:11px;border-left:3px solid {LG_HDR}}}
.sep-nl td{{background:{NL_SEP};color:{NL_HDR};font-weight:bold;
   padding:3px 10px;font-size:11px;border-left:3px solid {NL_HDR}}}
.sep-tot td{{background:{TOT_SEP};color:{TOT_FG};font-weight:bold;
   padding:3px 10px;font-size:11px;border-left:3px solid {TOT_HDR}}}
.r-tot td{{background:{TOT_ROW}!important;color:{TOT_FG}!important;font-weight:bold!important}}
.met{{background:{MET_BG}!important;color:{MET_FG}!important;font-weight:bold!important}}
.warn{{background:{WARN_BG}!important;color:{WARN_FG}!important;font-weight:bold!important}}
.miss{{background:{MISS_BG}!important;color:{MISS_FG}!important;font-weight:bold!important}}
.fail-red{{background:{MISS_BG}!important;color:{MISS_FG}!important;font-weight:bold!important}}
.fail-grn{{background:{MET_BG}!important;color:{MET_FG}!important;font-weight:bold!important}}
.sec-badge{{display:inline-block;font-size:12px;font-weight:bold;
   background:{SEC_BADGE_BG};color:{SEC_BADGE_FG};
   padding:4px 12px;margin:24px 0 4px;border-radius:3px}}
.note{{font-size:10.5px;color:#555;background:#f8f8f8;
   border-left:3px solid {SEC_BADGE_BG};padding:4px 10px;
   margin:0 0 10px;border-radius:0 3px 3px 0}}
"""

# ══════════════════════════════════════════════════════════════════════════════
# HTML HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def _th(l, bg=HDR_MID):
    return f'<th style="{TH_S}background:{bg};">{l}</th>'

def _thl(l, bg=HDR_DARK):
    return f'<th style="{TH_S}background:{bg};text-align:left;">{l}</th>'

def _sec(num, title, note, em=False):
    bs = (f"display:inline-block;font-size:12px;font-weight:bold;"
          f"background:{SEC_BADGE_BG};color:{SEC_BADGE_FG};"
          f"padding:4px 12px;margin:24px 0 4px;border-radius:3px;")
    ns = (f"{FONT}font-size:10.5px;color:#555;background:#f8f8f8;"
          f"border-left:3px solid {SEC_BADGE_BG};padding:4px 10px;"
          f"margin:0 0 10px;border-radius:0 3px 3px 0;display:block;")
    if em:
        return (f'<p style="margin:24px 0 4px;">'
                f'<span style="{bs}">{num}. {title}</span></p>'
                f'<p style="{ns}">{note}</p>')
    return (f'<div style="margin:24px 0 4px;">'
            f'<span class="sec-badge">{num}. {title}</span></div>'
            f'<div class="note">{note}</div>')

def spacer(em=False):
    if em:
        return ('<table width="100%" border="0" cellspacing="0" cellpadding="0">'
                '<tr><td style="height:24px;font-size:1px;">&nbsp;</td></tr></table>')
    return '<div style="height:24px;"></div>'

def lgd(inline=False):
    sp = "padding:2px 8px;margin-right:8px;font-weight:bold;"
    fs = f"{FONT}font-size:11px;"
    if inline:
        return (f'<p style="{fs}margin:6px 0 16px 0;">'
                f'<span style="{sp}background:{MET_BG};color:{MET_FG}">&#9632; Met(&ge;{IC_TARGET:.0f}%)</span>'
                f'<span style="{sp}background:{WARN_BG};color:{WARN_FG}">&#9632; Near</span>'
                f'<span style="{sp}background:{MISS_BG};color:{MISS_FG}">&#9632; Miss</span></p>')
    return (f'<div style="{fs}padding:8px 0 12px;">'
            f'<span style="{sp}background:{MET_BG};color:{MET_FG}">&#9632; Met(&ge;{IC_TARGET:.0f}%)</span>'
            f'<span style="{sp}background:{WARN_BG};color:{WARN_FG}">&#9632; Near</span>'
            f'<span style="{sp}background:{MISS_BG};color:{MISS_FG}">&#9632; Miss</span></div>')

# ══════════════════════════════════════════════════════════════════════════════
# COLORING HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def color_ic(v, em=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return ("","")
    fv = float(v)
    if fv >= IC_TARGET:         cls, bg, fg = "met",  MET_BG,  MET_FG
    elif fv >= IC_TARGET * 0.9: cls, bg, fg = "warn", WARN_BG, WARN_FG
    else:                       cls, bg, fg = "miss", MISS_BG, MISS_FG
    if em: return ("", f"background:{bg};color:{fg};font-weight:bold;")
    return (cls, "")

def color_sa(v, em=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return ("","")
    fv = float(v)
    if fv >= SA_TARGET:         cls, bg, fg = "met",  MET_BG,  MET_FG
    elif fv >= SA_TARGET * 0.9: cls, bg, fg = "warn", WARN_BG, WARN_FG
    else:                       cls, bg, fg = "miss", MISS_BG, MISS_FG
    if em: return ("", f"background:{bg};color:{fg};font-weight:bold;")
    return (cls, "")

def color_fail(v, em=False):
    if v is None or (isinstance(v, float) and np.isnan(v)): return ("","")
    cls = "fail-red" if int(v) > IC_FAIL_RED else "fail-grn"
    bg  = MISS_BG if int(v) > IC_FAIL_RED else MET_BG
    fg  = MISS_FG if int(v) > IC_FAIL_RED else MET_FG
    if em: return ("", f"background:{bg};color:{fg};font-weight:bold;")
    return (cls, "")

def color_sa_detail(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "#fff", "#000"
    fv = float(v)
    if fv >= 95:         return MET_BG,  MET_FG
    elif fv >= 95 * 0.9: return WARN_BG, WARN_FG
    else:                return MISS_BG, MISS_FG

# ══════════════════════════════════════════════════════════════════════════════
# FORMAT VALUE
# ══════════════════════════════════════════════════════════════════════════════
PCT_METRICS = {"IC %", "Staffing Attainment %"}
INT_METRICS = {"IC Pass", "IC Fail"}

def fmt_val(metric, v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return "&#8212;"
    if metric in PCT_METRICS: return f"{float(v):.1f}%"
    if metric in INT_METRICS: return f"{int(v):,}"
    return str(v)

# ══════════════════════════════════════════════════════════════════════════════
# PIVOT TABLE RENDERER
# ══════════════════════════════════════════════════════════════════════════════
LOB_STYLE = {
    "LG Chat": (LG_ROW, LG_HDR, LG_SEP, "sep-lg"),
    "NL Chat": (NL_ROW, NL_HDR, NL_SEP, "sep-nl"),
    "Total":   (TOT_ROW, TOT_FG, TOT_SEP, "sep-tot"),
}

def render_pivot_table(piv: pd.DataFrame, periods: list,
                       is_daily: bool = False,
                       show_total: bool = True,
                       em: bool = False) -> str:
    metrics  = piv["Metric"].unique().tolist()
    lob_keys = ["LG Chat", "NL Chat"] + (["Total"] if show_total else [])
    tc       = "" if em else 'class="t" '

    h = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}">']
    h.append('<thead><tr>')
    h.append(_thl("LOB"))
    h.append(_thl("Metric"))
    for p in periods:
        h.append(_th(p))
    h.append('</tr></thead><tbody>')

    prev_lob = None
    for lob in lob_keys:
        row_bg, lob_col, sep_bg, sep_cls = LOB_STYLE.get(lob, (TOT_ROW, TOT_FG, TOT_SEP, "sep-tot"))
        is_tot = lob == "Total"

        if lob != prev_lob:
            prev_lob = lob
            span = 2 + len(periods)
            if em:
                h.append(f'<tr><td colspan="{span}" style="{FONT}background:{sep_bg};'
                         f'color:{lob_col};font-weight:bold;padding:4px 12px;'
                         f'border-left:3px solid {lob_col}">{lob}</td></tr>')
            else:
                h.append(f'<tr class="{sep_cls}"><td colspan="{span}">{lob}</td></tr>')

        for mk in metrics:
            sub = piv[(piv["LOB"] == lob) & (piv["Metric"] == mk)]
            if sub.empty: continue

            if mk == "IC %":                   cfn = color_ic
            elif mk == "Staffing Attainment %": cfn = color_sa
            elif mk == "IC Fail" and is_daily:  cfn = color_fail
            else:                              cfn = None

            tr_cls = "r-tot" if is_tot else ""

            if em:
                h.append('<tr>')
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:left;'
                         f'color:{lob_col};font-weight:{"bold" if is_tot else "normal"}">{lob}</td>')
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:left;">{mk}</td>')
                for p in periods:
                    v   = sub.iloc[0].get(p)
                    val = fmt_val(mk, v)
                    if cfn:
                        inl = cfn(v, em=True)[1]
                        h.append(f'<td style="{TD_S}{inl}text-align:right;">{val}</td>')
                    else:
                        h.append(f'<td style="{TD_S}background:{row_bg};text-align:right;">{val}</td>')
                h.append('</tr>')
            else:
                h.append(f'<tr class="{tr_cls}">')
                h.append(f'<td class="lbl" style="{TD_S}background:{row_bg};'
                         f'color:{lob_col};font-weight:bold">{lob}</td>')
                h.append(f'<td class="lbl" style="{TD_S}background:{row_bg}">{mk}</td>')
                for p in periods:
                    v   = sub.iloc[0].get(p)
                    val = fmt_val(mk, v)
                    if cfn:
                        cls = cfn(v)[0]
                        h.append(f'<td class="{cls}" style="{TD_S}text-align:right;'
                                 f'background:{row_bg}">{val}</td>')
                    else:
                        h.append(f'<td style="{TD_S}background:{row_bg};text-align:right;">{val}</td>')
                h.append('</tr>')

    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# EXCALIBUR FAILED TABLE RENDERER (Section 2)
# Site columns: Req (MSP planned heads) | Act (actual heads) | Var (variance)
# Var > 0 green | Var < 0 red | Var = 0 amber
# Act vs Req: > green | < red | ~equal yellow
# ══════════════════════════════════════════════════════════════════════════════
def render_excalibur_fail_table(df_fail_in: pd.DataFrame, em: bool = False) -> str:
    if df_fail_in.empty:
        return (f'<p style="{FONT}color:{MET_FG};background:{MET_BG};'
                f'padding:8px 12px;border-radius:4px;">No failed intervals.</p>')

    COLS = (
        [
            ("Date_str",               "Date"),
            ("LOB_disp",               "LOB"),
            ("PST_Interval_Range",     "PST Interval"),
            ("VNT_Interval",           "VNT Interval"),
            ("Total PSP",              "Forecast Prod"),
            ("Total Productive Hours", "Total Prod Hrs"),
            ("SA %",                   "SA %"),
        ]
        + [
            col
            for site in SITES
            for col in [
                (f"{site}_raw_fcst", f"{site} Req"),
                (f"{site}_raw_prod", f"{site} Act"),
                (site,               f"{site} Var"),
            ]
        ]
    )

    def _col_hdr_bg(col_key: str) -> str:
        if col_key in SITES:
            return SITE_COLORS[col_key]["hdr"]
        if col_key.endswith("_raw_fcst"):
            return SITE_COLORS.get(col_key.replace("_raw_fcst",""), {}).get("bar", HDR_MID)
        if col_key.endswith("_raw_prod"):
            return SITE_COLORS.get(col_key.replace("_raw_prod",""), {}).get("hdr", HDR_DARK)
        if col_key in ("Total PSP", "Total Productive Hours", "SA %"):
            return "#4a5568"
        return HDR_DARK

    tc = "" if em else 'class="t" '
    h  = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}">']

    h.append('<thead><tr>')
    for col_key, col_hdr in COLS:
        bg    = _col_hdr_bg(col_key)
        align = ("left" if col_key in
                 ("Date_str","LOB_disp","PST_Interval_Range","VNT_Interval")
                 else "center")
        h.append(f'<th style="{TH_S}background:{bg};text-align:{align};">{col_hdr}</th>')
    h.append('</tr>')

    sub_s = (f"{FONT}background:#2d3748;color:#e2e8f0;font-weight:bold;"
             f"font-size:10px;padding:3px 8px;text-align:center;"
             f"border:1px solid rgba(255,255,255,0.15);")
    h.append('<tr>')
    h.append(f'<td colspan="7" style="{sub_s}text-align:left;">Info</td>')
    for site in SITES:
        bg = SITE_COLORS.get(site, {}).get("hdr", HDR_DARK)
        h.append(f'<td colspan="3" style="{sub_s}background:{bg};">{site}: Req | Act | Var</td>')
    h.append('</tr></thead>')

    prev_date = None
    for _, row in df_fail_in.iterrows():
        lob_disp = row.get("LOB_disp", "")
        date_str = row.get("Date_str", "")
        row_bg   = LG_ROW if lob_disp == "LG Chat" else NL_ROW
        lob_col  = LG_HDR if lob_disp == "LG Chat" else NL_HDR

        if date_str != prev_date:
            prev_date = date_str
            h.append(f'<tr><td colspan="{len(COLS)}" style="{FONT}background:#1E1E2E;'
                     f'color:#E8E8FF;font-weight:bold;padding:5px 12px;">{date_str}</td></tr>')

        h.append('<tr>')
        for col_key, _ in COLS:
            v       = row.get(col_key)
            is_null = v is None or (isinstance(v, float) and pd.isna(v))

            if col_key == "Date_str":
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:left;">{date_str}</td>')

            elif col_key == "LOB_disp":
                if lob_disp == "LG Chat":
                    bg_c, fg_c = "#1a6b3a", "#ffffff"
                else:
                    bg_c, fg_c = "#0e3d7a", "#ffffff"
                badge = (f'<span style="background:{bg_c};color:{fg_c};'
                         f'padding:2px 8px;border-radius:3px;'
                         f'font-weight:bold;font-size:11px;'
                         f'white-space:nowrap;">{lob_disp}</span>')
                h.append(f'<td style="{TD_S}background:#fff;text-align:left;">{badge}</td>')

            elif col_key in ("PST_Interval_Range", "VNT_Interval"):
                val = str(v) if not is_null else "&#8212;"
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:left;">{val}</td>')

            elif col_key == "SA %":
                if is_null:
                    val = "&#8212;"; bg_c, fg_c = row_bg, "#000"
                else:
                    fv_ = float(v); val = f"{fv_:.1f}%"
                    bg_c, fg_c = color_sa_detail(fv_)
                h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                         f'font-weight:bold;text-align:right;">{val}</td>')

            elif col_key in ("Total PSP", "Total Productive Hours"):
                val = f"{float(v):.2f}" if not is_null else "&#8212;"
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:right;">{val}</td>')

            elif col_key in SITES:
                if is_null:
                    val = "&#8212;"; bg_c, fg_c, fw = row_bg, "#000", "normal"
                else:
                    fv_ = float(v); fw = "bold"
                    val = f"+{fv_:.2f}" if fv_ > 0 else f"{fv_:.2f}"
                    if fv_ > 0:   bg_c, fg_c = "#1e8449", "#ffffff"
                    elif fv_ < 0: bg_c, fg_c = "#c0392b", "#ffffff"
                    else:         bg_c, fg_c = "#d4ac0d", "#1a1a1a"
                h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                         f'font-weight:{fw};text-align:right;'
                         f'border:1px solid rgba(0,0,0,0.15);">{val}</td>')

            elif col_key.endswith("_raw_fcst"):
                val = f"{float(v):.2f}" if not is_null else "&#8212;"
                h.append(f'<td style="{TD_S}background:{row_bg};color:#555;text-align:right;">{val}</td>')

            elif col_key.endswith("_raw_prod"):
                site  = col_key.replace("_raw_prod", "")
                req_v = row.get(f"{site}_raw_fcst")
                req_ok = req_v is not None and not (isinstance(req_v, float) and pd.isna(req_v))
                if is_null:
                    val = "&#8212;"; bg_c, fg_c = row_bg, "#000"; fw = "normal"
                else:
                    fv_ = float(v); val = f"{fv_:.2f}"; fw = "bold"
                    if not req_ok:
                        bg_c, fg_c = row_bg, "#000"
                    elif abs(fv_ - float(req_v)) < 0.01:
                        bg_c, fg_c = WARN_BG, WARN_FG
                    elif fv_ > float(req_v):
                        bg_c, fg_c = MET_BG,  MET_FG
                    else:
                        bg_c, fg_c = MISS_BG, MISS_FG
                h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                         f'font-weight:{fw};text-align:right;">{val}</td>')

        h.append('</tr>')

    h.append('</tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# IC DETAILS TABLE RENDERER (Section 3)
# Coloring rules:
#   Scheduled OpenTime < Req Heads -> red
#   Actual Heads < Scheduled OpenTime -> red, else green
#   Actual Break/Lunch/TC >= Scheduled -> red, else green
#   Scheduled Leave/NCNS/Termination > 0 -> red
#   Lost Heads (lateness) >= 0.5 -> solid red, else plain
# ══════════════════════════════════════════════════════════════════════════════
IC_DETAIL_COLS = [
    ("LOB_disp",                    "LOB"),
    ("Date_str",                    "PST Date"),
    ("PST_Interval_Range",          "PST Interval"),
    ("VNT_Date_str",                "VNT Date"),
    ("VNT_Interval_Range",          "VNT Interval"),
    ("IC Status",                   "IC Status"),
    ("SA_pct",                      "SA %"),
    ("Site Req Heads",              "Req Heads"),
    ("Scheduled_Open_Time",         "Scheduled OpenTime"),
    ("Site Actual Heads",           "Actual Heads"),
    ("Lost_Heads",                  "Lateness (Lost Heads)"),  # FIX: was "Lateness"
    ("Scheduled_Break",             "Scheduled Break"),
    ("Actual_Break",                "Actual Break"),
    ("Scheduled_Lunch",             "Scheduled Lunch"),
    ("Actual_Lunch",                "Actual Lunch"),
    ("Scheduled_Training/Coaching", "Scheduled Training/Coaching"),
    ("Actual_Training_Coaching",    "Actual Training/Coaching"),
    ("Scheduled_Leave",             "Scheduled Leave (AL+CO)"),
    ("Scheduled_NCNS",              "Scheduled NCNS"),
    ("Scheduled_Termination",       "Scheduled Termination"),
]

IC_HDR_COLORS = {
    "LOB":                          HDR_DARK,
    "PST Date":                     HDR_DARK,
    "PST Interval":                 HDR_DARK,
    "VNT Date":                     HDR_DARK,
    "VNT Interval":                 HDR_DARK,
    "IC Status":                    "#c0003c",
    "SA %":                         "#5a3e00",
    "Req Heads":                    "#4a5568",
    "Scheduled OpenTime":           "#1a6b3a",
    "Actual Heads":                 "#1a6b3a",
    "Lateness (Lost Heads)":        "#7b241c",  # FIX: key must match col_hdr
    "Scheduled Break":              "#7a5200",
    "Actual Break":                 "#7a5200",
    "Scheduled Lunch":              "#7a5200",
    "Actual Lunch":                 "#7a5200",
    "Scheduled Training/Coaching":  "#7a5200",
    "Actual Training/Coaching":     "#7a5200",
    "Scheduled Leave (AL+CO)":      "#5b2c6f",
    "Scheduled NCNS":               "#5b2c6f",
    "Scheduled Termination":        "#5b2c6f",
}

IC_SUM_COLS = [
    "Site Req Heads", "Scheduled_Open_Time", "Site Actual Heads", "Lost_Heads",
    "Scheduled_Break", "Actual_Break", "Scheduled_Lunch", "Actual_Lunch",
    "Scheduled_Training/Coaching", "Actual_Training_Coaching",
    "Scheduled_Leave", "Scheduled_NCNS", "Scheduled_Termination",
]

COLORED_COLS = {
    "Scheduled_Open_Time",
    "Site Actual Heads",
    "Actual_Break", "Actual_Lunch",
    "Actual_Training_Coaching",
    "Scheduled_Leave", "Scheduled_NCNS",
    "Scheduled_Termination",
}

def _ic_cell_color(col_key: str, v, row: pd.Series, row_bg: str):
    is_null = v is None or (isinstance(v, float) and pd.isna(v))
    if is_null:
        return row_bg, "#000", "normal"
    fv = float(v)

    if col_key == "Scheduled_Open_Time":
        req = row.get("Site Req Heads")
        if req is not None and not (isinstance(req, float) and pd.isna(req)):
            if fv < float(req): return "#c0392b", "#ffffff", "bold"
        return row_bg, "#000", "normal"

    if col_key == "Site Actual Heads":
        sch = row.get("Scheduled_Open_Time")
        if sch is not None and not (isinstance(sch, float) and pd.isna(sch)):
            if fv < float(sch): return "#c0392b", "#ffffff", "bold"
            else:               return "#1e8449", "#ffffff", "bold"
        return row_bg, "#000", "normal"

    if col_key == "Actual_Break":
        sch = row.get("Scheduled_Break")
        if sch is not None and not (isinstance(sch, float) and pd.isna(sch)):
            if fv >= float(sch): return "#c0392b", "#ffffff", "bold"
            else:                return "#1e8449", "#ffffff", "bold"
        return row_bg, "#000", "normal"

    if col_key == "Actual_Lunch":
        sch = row.get("Scheduled_Lunch")
        if sch is not None and not (isinstance(sch, float) and pd.isna(sch)):
            if fv >= float(sch): return "#c0392b", "#ffffff", "bold"
            else:                return "#1e8449", "#ffffff", "bold"
        return row_bg, "#000", "normal"

    if col_key == "Actual_Training_Coaching":
        sch = row.get("Scheduled_Training/Coaching")
        if sch is not None and not (isinstance(sch, float) and pd.isna(sch)):
            if fv >= float(sch): return "#c0392b", "#ffffff", "bold"
            else:                return "#1e8449", "#ffffff", "bold"
        return row_bg, "#000", "normal"

    if col_key in ("Scheduled_Leave", "Scheduled_NCNS", "Scheduled_Termination"):
        if fv > 0: return "#c0392b", "#ffffff", "bold"
        return row_bg, "#000", "normal"

    return row_bg, "#000", "normal"


def render_ic_details_table(df_in: pd.DataFrame, em: bool = False) -> str:
    if df_in.empty:
        return (f'<p style="{FONT}color:{MET_FG};background:{MET_BG};'
                f'padding:8px 12px;border-radius:4px;">No missed intervals.</p>')

    tc = "" if em else 'class="t" '
    h  = [f'<table {tc}style="border-collapse:collapse;width:auto;{FONT}">']
    h.append('<thead><tr>')
    for col_key, col_hdr in IC_DETAIL_COLS:
        bg = IC_HDR_COLORS.get(col_hdr, HDR_DARK)
        h.append(f'<th style="{TH_S}background:{bg};">{col_hdr}</th>')
    h.append('</tr></thead><tbody>')

    prev_date = None
    prev_lob  = None

    for _, row in df_in.iterrows():
        lob_disp = row.get("LOB_disp","")
        date_str = row.get("Date_str","")
        row_bg   = LG_ROW if lob_disp == "LG Chat" else NL_ROW
        lob_col  = LG_HDR if lob_disp == "LG Chat" else NL_HDR
        span     = len(IC_DETAIL_COLS)

        if date_str != prev_date:
            prev_date = date_str; prev_lob = None
            h.append(f'<tr><td colspan="{span}" style="{FONT}background:#1E1E2E;'
                     f'color:#E8E8FF;font-weight:bold;padding:5px 12px;">{date_str}</td></tr>')

        if lob_disp != prev_lob:
            prev_lob = lob_disp
            sep_bg   = LG_SEP if lob_disp == "LG Chat" else NL_SEP
            h.append(f'<tr><td colspan="{span}" style="{FONT}background:{sep_bg};'
                     f'color:{lob_col};font-weight:bold;padding:3px 12px;'
                     f'border-left:3px solid {lob_col}">{lob_disp}</td></tr>')

        h.append('<tr>')
        for col_key, col_hdr in IC_DETAIL_COLS:
            v       = row.get(col_key)
            is_null = v is None or (isinstance(v, float) and pd.isna(v))

            if col_key == "IC Status":
                ic_val = str(v) if not is_null else "&#8212;"
                bg_c   = MISS_BG if ic_val == "Missed" else MET_BG
                fg_c   = MISS_FG if ic_val == "Missed" else MET_FG
                h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                         f'font-weight:bold;text-align:center;">{ic_val}</td>')

            elif col_key == "SA_pct":
                if is_null:
                    val = "&#8212;"; bg_c, fg_c = row_bg, "#000"
                else:
                    val = f"{float(v):.1f}%"
                    bg_c, fg_c = color_sa_detail(float(v))
                h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                         f'font-weight:bold;text-align:right;">{val}</td>')

            elif col_key in ("Date_str","PST_Interval_Range","VNT_Date_str","VNT_Interval_Range"):
                val = str(v) if not is_null else "&#8212;"
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:left;'
                         f'color:#000;font-weight:normal;">{val}</td>')

            elif col_key == "LOB_disp":
                bg_c, fg_c = ("#1a6b3a","#ffffff") if lob_disp == "LG Chat" else ("#0e3d7a","#ffffff")
                badge = (f'<span style="background:{bg_c};color:{fg_c};'
                         f'padding:2px 8px;border-radius:3px;'
                         f'font-weight:bold;font-size:11px;'
                         f'white-space:nowrap;">{lob_disp}</span>')
                h.append(f'<td style="{TD_S}background:#fff;text-align:left;">{badge}</td>')

            elif col_key in COLORED_COLS:
                bg_c, fg_c, fw = _ic_cell_color(col_key, v, row, row_bg)
                val = f"{float(v):.1f}" if not is_null else "&#8212;"
                h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                         f'font-weight:{fw};text-align:right;">{val}</td>')

            elif col_key == "Lost_Heads":
                if is_null:
                    h.append(f'<td style="{TD_S}background:{row_bg};text-align:right;">&#8212;</td>')
                else:
                    fv_ = float(v)
                    if fv_ < 0.5:
                        h.append(f'<td style="{TD_S}background:{row_bg};'
                                 f'text-align:right;color:#000;">0.0</td>')
                    else:
                        h.append(f'<td style="{TD_S}background:#c0392b;color:#ffffff;'
                                 f'font-weight:bold;text-align:right;">{fv_:.1f}</td>')

            else:
                val = f"{float(v):.1f}" if not is_null else "&#8212;"
                h.append(f'<td style="{TD_S}background:{row_bg};text-align:right;">{val}</td>')

        h.append('</tr>')

    # FIX: Grand Total indentation was 3 spaces, now consistent 4 spaces
    gt_sa = (df_in["Staffing Attainment (Pct)"].mean() * 100) if not df_in.empty else None
    h.append('<tr>')
    for col_key, _ in IC_DETAIL_COLS:
        if col_key == "LOB_disp":
            h.append(f'<td style="{TD_S}background:{TOT_SEP};color:{TOT_FG};'
                     f'font-weight:bold;text-align:left;">Grand Total</td>')
        elif col_key == "IC Status":
            h.append(f'<td style="{TD_S}background:{MISS_BG};color:{MISS_FG};'
                     f'font-weight:bold;text-align:center;">Missed</td>')
        elif col_key == "SA_pct":
            val = f"{gt_sa:.1f}%" if gt_sa is not None else "&#8212;"
            bg_c, fg_c = color_sa_detail(gt_sa) if gt_sa is not None else (TOT_SEP, TOT_FG)
            h.append(f'<td style="{TD_S}background:{bg_c};color:{fg_c};'
                     f'font-weight:bold;text-align:right;">{val}</td>')
        elif col_key in IC_SUM_COLS and col_key in df_in.columns:
            total = pd.to_numeric(df_in[col_key], errors="coerce").sum()
            h.append(f'<td style="{TD_S}background:{TOT_SEP};color:{TOT_FG};'
                     f'font-weight:bold;text-align:right;">{total:.1f}</td>')
        else:
            h.append(f'<td style="{TD_S}background:{TOT_SEP};">&nbsp;</td>')
    h.append('</tr></tbody></table>')
    return "".join(h)

# ══════════════════════════════════════════════════════════════════════════════
# IC FAILURE ROOT CAUSE SUMMARY (below Section 3 table)
# ══════════════════════════════════════════════════════════════════════════════
def render_ic_summary(df_in: pd.DataFrame, em: bool = False) -> str:
    if df_in.empty:
        return ""

    FONT_S  = f"{FONT}font-size:11px;"
    parts   = []
    title_s = (f"{FONT_S}font-weight:bold;font-size:12px;"
               f"color:{HDR_DARK};margin:16px 0 8px;")
    parts.append(f'<p style="{title_s}">IC Failure Root Cause Summary</p>')

    prev_date = None

    for _, row in df_in.iterrows():
        date_str  = row.get("Date_str", "")
        lob_disp  = row.get("LOB_disp", "")
        pst_range = row.get("PST_Interval_Range", "")
        vnt_range = row.get("VNT_Interval_Range", "")
        lob_bg    = "#1a6b3a" if lob_disp == "LG Chat" else "#0e3d7a"

        reasons = []

        def _safe(col):
            v = row.get(col)
            return None if (v is None or (isinstance(v, float) and pd.isna(v))) else float(v)

        req     = _safe("Site Req Heads")
        sch_ot  = _safe("Scheduled_Open_Time")
        act_h   = _safe("Site Actual Heads")
        sch_brk = _safe("Scheduled_Break");          act_brk = _safe("Actual_Break")
        sch_lnc = _safe("Scheduled_Lunch");          act_lnc = _safe("Actual_Lunch")
        sch_tc  = _safe("Scheduled_Training/Coaching"); act_tc = _safe("Actual_Training_Coaching")
        ncns    = _safe("Scheduled_NCNS")
        term    = _safe("Scheduled_Termination")
        leave   = _safe("Scheduled_Leave")
        lost_h  = _safe("Lost_Heads")

        if req is not None and sch_ot is not None:
            diff = req - sch_ot
            if diff > 0:
                reasons.append(
                    f'scheduled short by <b>{diff:.1f}</b> heads vs Req '
                    f'({sch_ot:.1f} scheduled vs {req:.1f} required)')

        if sch_ot is not None and act_h is not None:
            diff = sch_ot - act_h
            if diff > 0:
                reasons.append(
                    f'actual short by <b>{diff:.1f}</b> heads vs scheduled '
                    f'({act_h:.1f} actual vs {sch_ot:.1f} scheduled)')

        if lost_h is not None and lost_h >= 0.5:
            reasons.append(f'<b>{lost_h:.1f}</b> heads lost to late arrival')

        if sch_brk is not None and act_brk is not None and (act_brk - sch_brk) > 0:
            reasons.append(f'<b>{act_brk - sch_brk:.1f}</b> unscheduled break heads')

        if sch_lnc is not None and act_lnc is not None and (act_lnc - sch_lnc) > 0:
            reasons.append(f'<b>{act_lnc - sch_lnc:.1f}</b> unscheduled lunch heads')

        if sch_tc is not None and act_tc is not None and (act_tc - sch_tc) > 0:
            reasons.append(f'<b>{act_tc - sch_tc:.1f}</b> unscheduled coaching/training heads')

        if ncns is not None and ncns > 0:
            reasons.append(f'<b>{ncns:.1f}</b> NCNS heads')

        if term is not None and term > 0:
            reasons.append(f'<b>{term:.1f}</b> Termination heads')

        if leave is not None and leave > 0:
            reasons.append(f'<b>{leave:.1f}</b> Leave (AL+CO) heads')

        if not reasons:
            continue

        if date_str != prev_date:
            prev_date = date_str
            parts.append(
                f'<p style="{FONT_S}background:#1E1E2E;color:#E8E8FF;'
                f'font-weight:bold;padding:4px 10px;margin:10px 0 4px;">'
                f'{date_str}</p>')

        lob_badge = (
            f'<span style="background:{lob_bg};color:#fff;'
            f'padding:1px 7px;border-radius:3px;'
            f'font-size:10px;font-weight:bold;">{lob_disp}</span>')

        parts.append(
            f'<p style="{FONT_S}margin:3px 0;line-height:1.8;'
            f'padding:5px 10px;border-left:3px solid {lob_bg};'
            f'background:#f8f9fa;">'
            f'{lob_badge} &nbsp;<b>{pst_range}</b>'
            f'<span style="color:#888;font-size:10px;"> (VNT: {vnt_range})</span>'
            f' — {" · ".join(reasons)}.</p>')

    return "".join(parts)

# ══════════════════════════════════════════════════════════════════════════════
# BUILD ALL SECTIONS
# ══════════════════════════════════════════════════════════════════════════════
def build_banner(em=False):
    bi  = f"{FONT}font-size:14px;font-weight:bold;color:#fff;margin:0;"
    bs  = f"{FONT}font-size:10.5px;color:#fff;margin:3px 0 0;"
    inn = (f'<p style="{bi}">&#128202; Expedia VN — IC &amp; Staffing Attainment Report</p>'
           f'<p style="{bs}">As of: <strong>{report_date_s}</strong> &nbsp;|&nbsp; '
           f'Generated: {report_now.strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp; '
           f'IC Target: &ge;{IC_TARGET:.0f}% &nbsp;|&nbsp; '
           f'SA Target: &ge;{SA_TARGET:.0f}%</p>')
    if em:
        return (f'<table width="100%" border="0" cellspacing="0" cellpadding="0" style="margin:0 0 14px;">'
                f'<tr><td style="background:{BANNER_C};padding:10px 14px;border-radius:4px;">'
                f'{inn}</td></tr></table>')
    return (f'<div style="background:{BANNER_C};padding:10px 14px;'
            f'border-radius:4px;margin:0 0 14px;">{inn}</div>')

def build_all(em=False):
    parts    = [build_banner(em)]
    d_range  = f"{daily_from} -> {daily_to}"

    parts.append(_sec("1", "Daily IC &amp; Staffing Attainment",
        f"IC % target &ge;{IC_TARGET:.0f}%. IC Fail: red if &gt;{IC_FAIL_RED}, green otherwise. "
        f"Range: {d_range}.", em))
    parts.append(render_pivot_table(daily_piv, daily_periods, is_daily=True, show_total=False, em=em))
    parts.append(lgd(inline=em))
    parts.append(spacer(em))

    parts.append(_sec("2", "Daily — Failed Intervals (Excalibur)",
        f"IC=0 from excalibur_raw. Forecast Productive = sum of MSP Site Wise. "
        f"SA: &ge;95% green | ~85.5% yellow | &lt;85.5% red. "
        f"Var = (Actual - Planned) heads: &gt;0 green | &lt;0 red | =0 amber. "
        f"Range: {d_range}.", em))
    parts.append(render_excalibur_fail_table(fail_site, em=em))
    parts.append(spacer(em))

    parts.append(_sec("3", "Daily — Failed Intervals Detail (IC HCM Log)",
        f"IC=Missed from IC_HCM_Details_Log. "
        f"Actual Heads: red if &lt; Scheduled OpenTime. "
        f"Actual Break/Lunch/T&amp;C: red if &ge; Scheduled. "
        f"Leave/NCNS/Termination: red if &gt; 0. "
        f"Lateness: heads lost to late productive start (&ge;0.5 shown in red). "
        f"Range: {d_range}.", em))
    parts.append(render_ic_details_table(df_ic_fail, em=em))
    #parts.append(render_ic_summary(df_ic_fail, em=em))
    parts.append(spacer(em))

    parts.append(_sec("4", f"Weekly IC &amp; Staffing Attainment — Last {N_WEEKLY_WEEKS} Weeks",
        f"IC Pass/Fail + IC % + SA % by week up to {daily_to}.", em))
    parts.append(render_pivot_table(weekly_piv, weekly_periods, is_daily=False, show_total=False, em=em))
    parts.append(lgd(inline=em))
    parts.append(spacer(em))

    parts.append(_sec("5", f"Monthly IC &amp; Staffing Attainment — Last {N_MONTHLY_MONTHS} Months",
        f"IC Pass/Fail + IC % + SA % by month up to {daily_to}.", em))
    parts.append(render_pivot_table(monthly_piv, monthly_periods, is_daily=False, show_total=False, em=em))
    parts.append(lgd(inline=em))

    return "".join(parts)

# ══════════════════════════════════════════════════════════════════════════════
# GREETING & SIGNATURE
# ══════════════════════════════════════════════════════════════════════════════
def greeting():
    return f"""
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">Dear Team,</p>
<p style="{FONT}font-size:12px;margin:0 0 10px;line-height:1.7">
    Please find the Expedia VN IC &amp; Staffing Attainment Report
    as of <strong>{report_date_s}</strong> ({daily_from} &#8594; {daily_to}).
</p>
<ul style="{FONT}font-size:12px;margin:0 0 16px;padding-left:20px;line-height:1.8">
    <li><strong>IC %</strong> — Pass / Total Intervals. Target &ge;{IC_TARGET:.0f}%</li>
    <li><strong>IC Fail</strong> — Red if &gt;{IC_FAIL_RED}, Green if &le;{IC_FAIL_RED} (daily only)</li>
    <li><strong>SA %</strong> — Total Productive Hrs / Total Forecast (MSP Site Wise sum)</li>
    <li><strong>Sec 2 Var cols</strong> — (Actual &#8722; Planned) heads: &gt;0 green | &lt;0 red | =0 amber</li>
    <li><strong>Sec 3 coloring</strong> — Actual Heads &lt; Scheduled OpenTime &#8594; red |
        Actual Break/Lunch/T&amp;C &ge; Scheduled &#8594; red |
        Leave/NCNS/Termination &gt; 0 &#8594; red |
        Lateness (Lost Heads) &ge; 0.5 &#8594; solid red</li>
    <li><span style="background:{MET_BG};color:{MET_FG};padding:1px 6px;font-weight:bold">Green</span> Met &nbsp;
        <span style="background:{WARN_BG};color:{WARN_FG};padding:1px 6px;font-weight:bold">Yellow</span> Near &nbsp;
        <span style="background:{MISS_BG};color:{MISS_FG};padding:1px 6px;font-weight:bold">Red</span> Miss</li>
</ul>
<p style="{FONT}font-size:12px;margin:0 0 6px;line-height:1.7"><strong>Data Sources:</strong></p>
<ul style="{FONT}font-size:12px;margin:0 0 16px;padding-left:20px;line-height:1.8">
    <li><strong>Forecast / Required Hours (Req)</strong> — Required Hours tab in the
        Performance All Site file from the latest Excalibur Performance emails.</li>
    <li><strong>Actual Productive Hours (Act)</strong> — Downloaded from the agent detail productive console.</li>
    <li><strong>HCM Site — Section 3</strong> — IC missed intervals with breakdown of headcount shortfall:
        Scheduled vs Actual Heads, Break, Lunch, Training/Coaching, Leave, NCNS, and Lateness.
        Agent Compliance Detail (attached) provides per-agent per-interval breakdown.</li>
</ul>
<hr style="border:none;border-top:1px solid #e0e0e0;margin:0 0 12px;">
"""

def signature():
    return f"""
<hr style="border:none;border-top:1px solid #e0e0e0;margin:12px 0 10px;">
<p style="{FONT}font-size:12px;margin:0 0 4px;">Thanks &amp; Regards,</p>
<p style="{FONT}font-size:12px;font-weight:bold;margin:0 0 2px;">Chinh Nguyen</p>
<p style="{FONT}font-size:11px;color:#555;font-weight:bold;margin:0 0 2px;">BI Associate</p>
<p style="{FONT}font-size:11px;color:#555;margin:0 0 2px;line-height:1.6">
    Level 4, Tower 1, OneHub Saigon, Lot C1-2, D1 Street, Saigon Hi Tech Park,<br>
    Tan Phu Ward, District 9, Ho Chi Minh City, Vietnam
</p>
<p style="{FONT}font-size:11px;color:#555;margin:0;">
    Ph No: +84 986 473 419 &nbsp;|&nbsp;
    Email: <a href="mailto:huuchinh.nguyen@concentrix.com"
       style="color:{HDR_MID};font-weight:bold;text-decoration:none;">
       huuchinh.nguyen@concentrix.com</a>
</p>
<p style="{FONT}font-size:10px;color:#aaa;margin-top:8px;">
    Generated: {report_now.strftime("%Y-%m-%d %H:%M")} &nbsp;|&nbsp;
    Source: excalibur_raw.parquet + IC_HCM_Details_Log.parquet
</p>
"""

# ══════════════════════════════════════════════════════════════════════════════
# DISPLAY NOTEBOOK
# ══════════════════════════════════════════════════════════════════════════════
if DISPLAY_NOTEBOOK:
    nb  = ("<!DOCTYPE html><html><head><meta charset='utf-8'>"
           f"<style>{CSS}</style></head><body>"
           + greeting() + build_all(em=False) + signature()
           + "</body></html>")
    esc = nb.replace("&","&amp;").replace('"',"&quot;").replace("'","&#39;")
    display(HTML(
        f'<iframe srcdoc="{esc}" style="width:100%;border:none;min-height:900px;" '
        f'onload="this.style.height=(this.contentDocument.body.scrollHeight+40)+\'px\'"></iframe>'))
    print("Display done")

# ══════════════════════════════════════════════════════════════════════════════
# SEND EMAIL
# FIX: attachment block moved inside if SEND_EMAIL to avoid NameError
# ══════════════════════════════════════════════════════════════════════════════
if SEND_EMAIL:
    import win32com.client

    html_body = (
        "<!--[if mso]><xml><o:OfficeDocumentSettings><o:AllowPNG/>"
        "<o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml><![endif]-->"
        f"<div style='padding:20px 24px;background:#fff;{FONT}'>"
        + greeting() + build_all(em=True) + signature() + "</div>"
    )

    ATTACHMENT_FOLDER   = (f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files"
                           f"/Rawdata/OUTPUT_NON_COMPLIANCE")
    ATTACHMENT_FILE     = os.path.join(ATTACHMENT_FOLDER, "Agent_Compliance_Detail_HCM.csv")
    ATTACHMENT_FILTERED = os.path.join(ATTACHMENT_FOLDER, "Agent_Compliance_Detail_HCM_filtered.csv")

    if os.path.exists(ATTACHMENT_FILE):
        df_attach = pd.read_csv(ATTACHMENT_FILE, encoding="utf-8-sig")
        df_attach["Date"] = pd.to_datetime(df_attach["Date"], errors="coerce").dt.normalize()
        df_attach = df_attach[
            (df_attach["Date"].dt.date >= daily_from) &
            (df_attach["Date"].dt.date <= daily_to)
        ]
        df_attach.to_csv(ATTACHMENT_FILTERED, index=False, encoding="utf-8-sig")
        print(f"Attachment filtered: {len(df_attach):,} rows ({daily_from} -> {daily_to})")
        attach_path = ATTACHMENT_FILTERED
    else:
        print(f"Attachment not found: {ATTACHMENT_FILE}")
        attach_path = None

    def send_auto(to, cc, subject, html_body, attachment_path=None, quit_after=True):
        pythoncom.CoInitialize()
        was_on = any(p.name().lower() == "outlook.exe"
                     for p in psutil.process_iter(["name"]))
        if not was_on:
            for exe in [
                r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE",
            ]:
                if os.path.exists(exe): subprocess.Popen([exe]); break
            print("Starting Outlook...")
            for _ in range(30):
                time.sleep(1)
                try: win32com.client.GetActiveObject("Outlook.Application"); break
                except: pass
        try:
            ol   = win32com.client.Dispatch("Outlook.Application")
            ns   = ol.GetNamespace("MAPI"); ns.Logon()
            mail = ol.CreateItem(0)
            mail.To       = to
            mail.CC       = cc
            mail.Subject  = subject
            mail.HTMLBody = html_body
            if attachment_path and os.path.exists(attachment_path):
                mail.Attachments.Add(os.path.abspath(attachment_path))
                print(f"Attached: {os.path.basename(attachment_path)}")
            mail.Send()
            print(f"Sent -> {to}")
            time.sleep(3)
        finally:
            if quit_after and not was_on:
                try: ol.Quit(); print("Outlook closed")
                except: pass

    send_auto(EMAIL_TO, EMAIL_CC, EMAIL_SUBJECT, html_body,
              attachment_path=attach_path,
              quit_after=True)

AUTO D-1 = 2026-07-24
Loading excalibur_raw.parquet...
excalibur: 70,560 rows | 2025-12-29 -> 2026-07-26
Loading IC_HCM_Details_Log.parquet...
IC Details: 18,203 rows | 2025-12-01 -> 2026-07-21
Excalibur max: 2026-07-26 -> daily_to: 2026-07-26
Daily range: 2026-07-01 -> 2026-07-26
Subject    : Expedia VN — IC & Staffing Attainment Report for LGC & NLC as of 24-Jul-2026
Building Daily pivot...
Daily periods: ['2026-07-01', '2026-07-02', '2026-07-03', '2026-07-04', '2026-07-05', '2026-07-06', '2026-07-07', '2026-07-08', '2026-07-09', '2026-07-10', '2026-07-11', '2026-07-12', '2026-07-13', '2026-07-14', '2026-07-15', '2026-07-16', '2026-07-17', '2026-07-18', '2026-07-19', '2026-07-20', '2026-07-21', '2026-07-22', '2026-07-23', '2026-07-24', '2026-07-25', '2026-07-26']
Building Excalibur failed intervals...
Excalibur failed intervals: 136
Building IC Details failed intervals...
IC Details failed intervals: 142
Building Weekly pivot...
Weekly (8): ['06/01~06/07', '06/08~06/14', '06/15~06/21

c:\Users\huuchinh.nguyen\AppData\Local\anaconda3\Lib\site-packages\IPython\core\display.py:431: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Display done
Attachment filtered: 42,410 rows (2026-07-01 -> 2026-07-26)
Attached: Agent_Compliance_Detail_HCM_filtered.csv
Sent -> puneet.suneja@concentrix.com;kirpan.patar@concentrix.com;ML.HOC.Expedia.Hierarchy@concentrix.com;EG_CAI_RAYAH_expedia_global_rtm@concentrix.com
